In [1]:
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
class RAFDBDataset:
    def __init__(self, root_dir, split='train'):
        self.root_dir = Path(root_dir) / split
        self.images = []
        self.labels = []
        self._load_data()
    
    def _load_data(self):
        for class_folder in sorted(self.root_dir.iterdir()):
            class_idx = int(class_folder.name)
            label = class_idx - 1
            for ext in ['*.jpg', '*.jpeg', '*.png']:
                for img_path in class_folder.glob(ext):
                    self.images.append(str(img_path))
                    self.labels.append(label)
        
        print(f"Загружено {len(self.images)} изображений")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

In [3]:
def create_fixed_split(root_dir, val_split=0.15, random_seed=42, save_path="dataset_splits.pkl"):
    full_train = RAFDBDataset(root_dir, split='train')
    all_indices = list(range(len(full_train)))
    all_labels = full_train.labels
    train_indices, val_indices = train_test_split(all_indices, test_size=val_split, random_state=random_seed, stratify=all_labels)
    test_dataset = RAFDBDataset(root_dir, split='test')
    test_indices = list(range(len(test_dataset)))
    splits = {'train_indices': train_indices, 'val_indices': val_indices, 'test_indices': test_indices, 'train_labels': [full_train.labels[i] for i in train_indices], 'val_labels': [full_train.labels[i] for i in val_indices], 'test_labels': test_dataset.labels, 'random_seed': random_seed, 'val_split': val_split}
    with open(save_path, 'wb') as f:
        pickle.dump(splits, f)
    print(f" Train: {len(train_indices)}  Val: {len(val_indices)}  Test: {len(test_indices)}")
    emotion_names = {0: 'Surprise', 1: 'Fear', 2: 'Disgust', 3: 'Happiness', 4: 'Sadness', 5: 'Anger', 6: 'Neutral'}
    print("\nРаспределение по классам:")
    for class_idx in range(7):
        name = emotion_names[class_idx]
        train_count = splits['train_labels'].count(class_idx)
        val_count = splits['val_labels'].count(class_idx)
        test_count = splits['test_labels'].count(class_idx)
        print(f" {name:9} train {train_count:4d}  val {val_count:3d}  test {test_count:3d}")
    return splits

In [4]:
def load_fixed_split(splits_path="dataset_splits.pkl"):
    with open(splits_path, 'rb') as f:
        splits = pickle.load(f)
    print(f" Train: {len(splits['train_indices']):5d}")
    print(f" Val:   {len(splits['val_indices']):5d}")
    print(f" Test:  {len(splits['test_indices']):5d}")
    return splits

In [5]:
RAFDB_ROOT = r"D:\НИР\RAF-DB\DATASET"
splits = create_fixed_split(root_dir=RAFDB_ROOT, val_split=0.15, random_seed=42, save_path="dataset_splits.pkl")
loaded_splits = load_fixed_split("dataset_splits.pkl")

Загружено 12271 изображений
Загружено 3068 изображений
 Train: 10430  Val: 1841  Test: 3068

Распределение по классам:
 Surprise  train 1097  val 193  test 329
 Fear      train  239  val  42  test  74
 Disgust   train  609  val 108  test 160
 Happiness train 4056  val 716  test 1185
 Sadness   train 1685  val 297  test 478
 Anger     train  599  val 106  test 162
 Neutral   train 2145  val 379  test 680
 Train: 10430
 Val:    1841
 Test:   3068
